In [3]:
# ── Imports ───────────────────────────────────────────────────────────
import os
import gc
import glob
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations
from joblib import Parallel, delayed
import pyarrow as pq

import lightgbm as lgb
from sklearn.model_selection import GroupKFold, KFold
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import shap
import psutil

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────
DATA_DIR        = Path(r"E:\Optiver\processed")
FOLDS           = [DATA_DIR / f"fold_{i}" for i in range(5)]
OUTPUT_DIR        = Path(r"E:\Optiver\outputs")
FEATURES_DIR      = OUTPUT_DIR / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Window config ─────────────────────────────────────────────────────
OBS_END         = 480
TARGET_START    = 480
TOTAL_SECONDS   = 600
BUCKET_SIZE     = 120
N_BUCKETS       = OBS_END // BUCKET_SIZE   # 4
EPS             = 1e-10
RV_FLOOR        = 1e-4

# ── Model config ──────────────────────────────────────────────────────
SEED            = 42
N_FOLDS_INNER   = 5
N_ROUNDS        = 5000
EARLY_STOPPING  = 200
ANCHOR_COL      = "past_log_rv_bkt3"

# ── Cluster mapping ───────────────────────────────────────────────────
STOCK_CLUSTER_MAP = {
    0:1, 1:1, 2:1, 3:1, 4:1, 5:1, 6:1, 7:1, 8:1, 9:1,
    10:1, 11:1, 13:0, 14:1, 15:1, 16:1, 17:1, 18:1, 19:1, 20:1,
    21:0, 22:1, 23:1, 26:1, 27:1, 28:1, 29:1, 30:1, 31:2, 32:0,
    33:1, 34:1, 35:2, 36:1, 37:1, 38:1, 39:1, 40:1, 41:0, 42:1,
    43:1, 44:2, 46:0, 47:0, 48:1, 50:1, 51:1, 52:1, 53:1, 55:1,
    56:1, 58:1, 59:1, 60:1, 61:1, 62:1, 63:2, 64:1, 66:1, 67:1,
    68:1, 69:1, 70:1, 72:1, 73:1, 74:2, 75:1, 76:1, 77:0, 78:1,
    80:1, 81:1, 82:1, 83:1, 84:1, 85:2, 86:2, 87:1, 88:1, 89:2,
    90:1, 93:1, 94:1, 95:1, 96:1, 97:1, 98:1, 99:2, 100:1, 101:1,
    102:1, 103:1, 104:1, 105:2, 107:1, 108:0, 109:1, 110:1, 111:0,
    112:1, 113:1, 114:1, 115:1, 116:1, 118:1, 119:2, 120:1, 122:1,
    123:1, 124:2, 125:0, 126:1,
}

CLUSTER_STOCKS = {0: [], 1: [], 2: []}
for stock, cluster in STOCK_CLUSTER_MAP.items():
    CLUSTER_STOCKS[cluster].append(stock)

np.random.seed(SEED)
print("Cell 1 complete — imports and config loaded")
print(f"Cluster sizes: { {c: len(s) for c, s in CLUSTER_STOCKS.items()} }")

Cell 1 complete — imports and config loaded
Cluster sizes: {0: 10, 1: 90, 2: 12}


In [7]:
# ── Fold paths (lazy — nothing loaded yet) ───────────────────────────
FOLD_PATHS = [(DATA_DIR / f"fold_{i}" / "train.parquet",
               DATA_DIR / f"fold_{i}" / "test.parquet")
              for i in range(5)]

# ── Only columns needed for feature engineering ───────────────────────
LOAD_COLS = [
    "stock_id", "time_id", "seconds_in_bucket",
    "wap", "bid_ask_spread", "total_volume",
    "price_spread", "depth_imbalance"
]

print("Cell 2 complete — fold paths registered, nothing loaded yet")
print(f"Columns to load per fold: {LOAD_COLS}")

Cell 2 complete — fold paths registered, nothing loaded yet
Columns to load per fold: ['stock_id', 'time_id', 'seconds_in_bucket', 'wap', 'bid_ask_spread', 'total_volume', 'price_spread', 'depth_imbalance']


In [8]:
# ── Helpers ───────────────────────────────────────────────────────────

def safe_log(x):
    return np.log(np.maximum(np.asarray(x, dtype=np.float64), EPS))

def vec_autocorr_lag1(values: np.ndarray, tid_mapped: np.ndarray,
                      n_tids: int) -> np.ndarray:
    """
    Vectorised lag-1 autocorrelation per time_id group.
    Returns array of length n_tids, one value per group.
    """
    out = np.zeros(n_tids, dtype=np.float64)
    for i in range(n_tids):
        mask = tid_mapped == i
        v = values[mask]
        if len(v) < 3:
            continue
        x, y   = v[:-1], v[1:]
        mx, my = x.mean(), y.mean()
        num    = ((x - mx) * (y - my)).sum()
        den    = np.sqrt(((x - mx)**2).sum() * ((y - my)**2).sum())
        out[i] = num / (den + EPS)
    return out

def vec_spread_trend(seconds: np.ndarray, spread: np.ndarray,
                     tid_mapped: np.ndarray, n_tids: int) -> np.ndarray:
    """
    Vectorised spread trend slope per time_id using analytic OLS.
    Returns array of length n_tids.
    """
    out = np.zeros(n_tids, dtype=np.float64)
    for i in range(n_tids):
        mask = tid_mapped == i
        t = seconds[mask].astype(np.float64)
        s = spread[mask].astype(np.float64)
        if len(t) < 3:
            continue
        t_mean = t.mean()
        s_mean = s.mean()
        var_t  = ((t - t_mean)**2).sum()
        if var_t < EPS:
            continue
        out[i] = ((t - t_mean) * (s - s_mean)).sum() / var_t
    return out

def vec_bpv(log_ret: np.ndarray, tid_mapped: np.ndarray,
            n_tids: int) -> pd.DataFrame:
    """
    Vectorised bipower variation and jump features per time_id.
    Returns DataFrame indexed 0..n_tids-1.
    """
    bpv_out        = np.zeros(n_tids)
    jump_out       = np.zeros(n_tids)
    log_bpv_out    = np.full(n_tids, np.log(RV_FLOOR))
    log_jump_out   = np.full(n_tids, np.log(RV_FLOOR))
    jump_ratio_out = np.zeros(n_tids)
    jump_frac_out  = np.zeros(n_tids)

    for i in range(n_tids):
        mask = tid_mapped == i
        r    = log_ret[mask]
        if len(r) < 2:
            continue
        rv   = float(np.sum(r**2))
        bpv  = float((np.pi / 2) * np.mean(np.abs(r[1:]) * np.abs(r[:-1])) * len(r))
        jump = max(rv - bpv, 0.0)
        bpv_out[i]        = np.sqrt(bpv)
        jump_out[i]       = np.sqrt(jump)
        log_bpv_out[i]    = float(safe_log([max(bpv,  RV_FLOOR)])[0])
        log_jump_out[i]   = float(safe_log([max(jump, RV_FLOOR)])[0])
        jump_ratio_out[i] = jump / (rv + EPS)
        jump_frac_out[i]  = 1.0 if jump > 0 else 0.0

    return pd.DataFrame({
        "bpv":        bpv_out,
        "jump":       jump_out,
        "log_bpv":    log_bpv_out,
        "log_jump":   log_jump_out,
        "jump_ratio": jump_ratio_out,
        "jump_frac":  jump_frac_out,
    })

def vec_semi(log_ret: np.ndarray, tid_mapped: np.ndarray,
             n_tids: int) -> pd.DataFrame:
    """
    Vectorised realized semivariance per time_id.
    Returns DataFrame indexed 0..n_tids-1.
    """
    rv_up_out   = np.zeros(n_tids)
    rv_down_out = np.zeros(n_tids)
    asym_out    = np.zeros(n_tids)

    for i in range(n_tids):
        mask    = tid_mapped == i
        r       = log_ret[mask]
        rv_up   = float(np.sum(r[r > 0]**2))
        rv_down = float(np.sum(r[r < 0]**2))
        rv_tot  = rv_up + rv_down + EPS
        rv_up_out[i]   = np.sqrt(rv_up)
        rv_down_out[i] = np.sqrt(rv_down)
        asym_out[i]    = (rv_up - rv_down) / rv_tot

    return pd.DataFrame({
        "rv_up":        rv_up_out,
        "rv_down":      rv_down_out,
        "log_rv_up":    np.log(np.clip(rv_up_out,   RV_FLOOR, None)),
        "log_rv_down":  np.log(np.clip(rv_down_out, RV_FLOOR, None)),
        "rv_asymmetry": asym_out,
        "signed_jump":  asym_out,
    })

def vec_spread_rv_corr(spread: np.ndarray, abs_ret: np.ndarray,
                       tid_mapped: np.ndarray, n_tids: int) -> np.ndarray:
    """
    Vectorised correlation between spread and abs_ret per time_id.
    Returns array of length n_tids.
    """
    out = np.zeros(n_tids, dtype=np.float64)
    for i in range(n_tids):
        mask = tid_mapped == i
        s = spread[mask][1:]
        r = abs_ret[mask][1:]
        if len(s) < 3:
            continue
        c = np.corrcoef(s, r)
        out[i] = float(c[0, 1]) if np.isfinite(c[0, 1]) else 0.0
    return out


# ── Main feature engineering function ────────────────────────────────

def process_one_stock(stock_df: pd.DataFrame) -> pd.DataFrame:
    stock_id = stock_df["stock_id"].iloc[0]

    df_obs = stock_df[stock_df["seconds_in_bucket"] < OBS_END].copy()
    df_tgt = stock_df[stock_df["seconds_in_bucket"] >= TARGET_START].copy()
    if df_obs.empty or df_tgt.empty:
        return pd.DataFrame()

    df_obs = df_obs.sort_values(["time_id", "seconds_in_bucket"])

    # ── Base transforms ───────────────────────────────────────────────
    df_obs["log_wap"]    = safe_log(df_obs["wap"].values)
    df_obs["log_ret"]    = df_obs.groupby("time_id")["log_wap"].diff().fillna(0)
    df_obs["abs_ret"]    = df_obs["log_ret"].abs()
    df_obs["sq_ret"]     = df_obs["log_ret"] ** 2
    df_obs["log_spread"] = safe_log(df_obs["bid_ask_spread"].values)
    df_obs["log_volume"] = safe_log(df_obs["total_volume"].clip(lower=1).values)
    df_obs["signed_vol"] = np.sign(df_obs["log_ret"].values) * df_obs["total_volume"].values
    df_obs["vw_ret"]     = df_obs["log_ret"].values * df_obs["total_volume"].values
    df_obs["vw_spread"]  = df_obs["bid_ask_spread"].values * df_obs["total_volume"].values

    # ── Group mappings ────────────────────────────────────────────────
    time_ids  = np.sort(df_obs["time_id"].unique())
    n_tids    = len(time_ids)
    tid_index = {t: i for i, t in enumerate(time_ids)}
    tid_mapped = np.array([tid_index[t] for t in df_obs["time_id"].values],
                          dtype=np.int32)
    g_obs = df_obs.groupby("time_id")

    # ── A. Interval RV lags (30s buckets) ─────────────────────────────
    n_intervals        = OBS_END // 30
    df_obs["interval"] = (df_obs["seconds_in_bucket"] // 30).clip(
                          upper=n_intervals - 1).astype(int)
    interval_rv = (
        df_obs.groupby(["time_id", "interval"])["log_ret"]
        .apply(lambda x: np.sqrt(np.sum(x**2)))
        .unstack(level="interval")
    ).fillna(0)
    interval_rv.columns = [f"rv_lag_{int(c)+1}" for c in interval_rv.columns]
    for c in list(interval_rv.columns):
        interval_rv[f"log_{c}"] = safe_log(interval_rv[c].values)

    # ── B. HAR aggregates ─────────────────────────────────────────────
    log_rv_cols  = sorted([c for c in interval_rv.columns
                           if c.startswith("log_rv_lag_")])
    n_lags       = len(log_rv_cols)
    short_cols   = [f"log_rv_lag_{i}" for i in range(n_lags - 2, n_lags + 1)
                    if f"log_rv_lag_{i}" in interval_rv]
    med_cols     = [f"log_rv_lag_{i}" for i in range(n_lags - 7, n_lags + 1)
                    if f"log_rv_lag_{i}" in interval_rv]
    interval_rv["log_rv_short"]   = interval_rv[short_cols].mean(axis=1) if short_cols else 0
    interval_rv["log_rv_medium"]  = interval_rv[med_cols].mean(axis=1)   if med_cols   else 0
    interval_rv["log_rv_long"]    = interval_rv[log_rv_cols].mean(axis=1)
    interval_rv["har_short_long"] = interval_rv["log_rv_short"]  - interval_rv["log_rv_long"]
    interval_rv["har_short_med"]  = interval_rv["log_rv_short"]  - interval_rv["log_rv_medium"]
    interval_rv["har_med_long"]   = interval_rv["log_rv_medium"] - interval_rv["log_rv_long"]
    if n_lags >= 3:
        X_idx  = np.arange(n_lags, dtype=np.float64)
        X_mean = X_idx.mean()
        X_var  = ((X_idx - X_mean)**2).sum()
        rv_v   = interval_rv[log_rv_cols].values
        Y_mean = rv_v.mean(axis=1, keepdims=True)
        interval_rv["rv_trend_slope"] = (
            (rv_v - Y_mean) * (X_idx - X_mean)
        ).sum(axis=1) / (X_var + EPS)
    interval_rv["rv_lag_std"] = interval_rv[log_rv_cols].std(axis=1)

    # ── C. Session RV at multiple windows ─────────────────────────────
    rv_full     = g_obs["sq_ret"].sum().apply(np.sqrt).rename("rv_full")
    rv_last60   = df_obs[df_obs["seconds_in_bucket"] >= 420].groupby(
                  "time_id")["sq_ret"].sum().apply(np.sqrt).rename("rv_last60")
    rv_last120  = df_obs[df_obs["seconds_in_bucket"] >= 360].groupby(
                  "time_id")["sq_ret"].sum().apply(np.sqrt).rename("rv_last120")
    rv_last240  = df_obs[df_obs["seconds_in_bucket"] >= 240].groupby(
                  "time_id")["sq_ret"].sum().apply(np.sqrt).rename("rv_last240")
    rv_first_h  = df_obs[df_obs["seconds_in_bucket"] <  240].groupby(
                  "time_id")["sq_ret"].sum().apply(np.sqrt).rename("rv_first_half")
    rv_second_h = df_obs[df_obs["seconds_in_bucket"] >= 240].groupby(
                  "time_id")["sq_ret"].sum().apply(np.sqrt).rename("rv_second_half")

    def make_log_rv(series, name):
        return pd.Series(safe_log(series.reindex(time_ids).fillna(0).values),
                         index=time_ids, name=name)

    log_rv_full     = make_log_rv(rv_full,     "log_rv_full")
    log_rv_last60   = make_log_rv(rv_last60,   "log_rv_last60")
    log_rv_last120  = make_log_rv(rv_last120,  "log_rv_last120")
    log_rv_last240  = make_log_rv(rv_last240,  "log_rv_last240")
    log_rv_first_h  = make_log_rv(rv_first_h,  "log_rv_first_half")
    log_rv_second_h = make_log_rv(rv_second_h, "log_rv_second_half")

    # ── D. RV ratios and acceleration ─────────────────────────────────
    rv_full_s     = rv_full.reindex(time_ids).fillna(0)
    rv_first_h_s  = rv_first_h.reindex(time_ids).fillna(0)
    rv_second_h_s = rv_second_h.reindex(time_ids).fillna(0)

    rv_ratio_60     = (rv_last60.reindex(time_ids).fillna(0)   / rv_full_s.clip(lower=EPS)).rename("rv_ratio_60")
    rv_ratio_120    = (rv_last120.reindex(time_ids).fillna(0)  / rv_full_s.clip(lower=EPS)).rename("rv_ratio_120")
    rv_ratio_240    = (rv_last240.reindex(time_ids).fillna(0)  / rv_full_s.clip(lower=EPS)).rename("rv_ratio_240")
    rv_ratio_halves = (rv_second_h_s / rv_first_h_s.clip(lower=EPS)).rename("rv_ratio_halves")
    log_rv_ratio_60  = pd.Series(safe_log(rv_ratio_60.values),
                                 index=time_ids, name="log_rv_ratio_60")
    log_rv_ratio_120 = pd.Series(safe_log(rv_ratio_120.values),
                                 index=time_ids, name="log_rv_ratio_120")
    rv_accel         = (rv_second_h_s - rv_first_h_s).rename("rv_accel")

    # ── E. Bipower variation and jump (vectorised) ────────────────────
    bpv_df = vec_bpv(df_obs["log_ret"].values, tid_mapped, n_tids)
    bpv_df.index = time_ids

    # ── F. Return distribution ────────────────────────────────────────
    ret_mean     = g_obs["log_ret"].mean().rename("ret_mean")
    ret_std      = g_obs["log_ret"].std().rename("ret_std")
    ret_skew     = g_obs["log_ret"].skew().rename("ret_skew")
    ret_kurt     = g_obs["log_ret"].apply(pd.Series.kurt).rename("ret_kurt")
    ret_min      = g_obs["log_ret"].min().rename("ret_min")
    ret_max      = g_obs["log_ret"].max().rename("ret_max")
    ret_range    = (ret_max - ret_min).rename("ret_range")
    ret_iqr      = (g_obs["log_ret"].quantile(0.75)
                   - g_obs["log_ret"].quantile(0.25)).rename("ret_iqr")
    abs_ret_mean = g_obs["abs_ret"].mean().rename("abs_ret_mean")
    abs_ret_max  = g_obs["abs_ret"].max().rename("abs_ret_max")
    abs_ret_p95  = g_obs["abs_ret"].quantile(0.95).rename("abs_ret_p95")

    # ── G. Autocorrelation lag-1 (vectorised) ─────────────────────────
    ret_ac1_vals     = vec_autocorr_lag1(df_obs["log_ret"].values,
                                         tid_mapped, n_tids)
    abs_ac1_vals     = vec_autocorr_lag1(df_obs["abs_ret"].values,
                                         tid_mapped, n_tids)
    ret_ac1          = pd.Series(ret_ac1_vals,  index=time_ids, name="ret_ac1")
    abs_ret_ac1      = pd.Series(abs_ac1_vals,  index=time_ids, name="abs_ret_ac1")

    # ── H. Spread features ────────────────────────────────────────────
    spread_mean   = g_obs["bid_ask_spread"].mean().rename("spread_mean")
    spread_std    = g_obs["bid_ask_spread"].std().rename("spread_std")
    spread_max    = g_obs["bid_ask_spread"].max().rename("spread_max")
    spread_min    = g_obs["bid_ask_spread"].min().rename("spread_min")
    spread_range  = (spread_max - spread_min).rename("spread_range")
    spread_cv     = (spread_std / spread_mean.clip(lower=EPS)).rename("spread_cv")
    spread_last60 = df_obs[df_obs["seconds_in_bucket"] >= 420].groupby(
                    "time_id")["bid_ask_spread"].mean().rename("spread_last60")
    spread_ratio  = (spread_last60.reindex(time_ids).fillna(0) /
                     spread_mean.clip(lower=EPS)).rename("spread_ratio")
    spread_skew   = g_obs["bid_ask_spread"].skew().rename("spread_skew")
    log_spread_mean = g_obs["log_spread"].mean().rename("log_spread_mean")
    log_spread_std  = g_obs["log_spread"].std().rename("log_spread_std")

    spread_trend_vals = vec_spread_trend(
        df_obs["seconds_in_bucket"].values,
        df_obs["bid_ask_spread"].values,
        tid_mapped, n_tids
    )
    spread_trend = pd.Series(spread_trend_vals, index=time_ids, name="spread_trend")

    spread_first60      = df_obs[df_obs["seconds_in_bucket"] < 60].groupby(
                          "time_id")["bid_ask_spread"].mean()
    spread_change_ratio = (
        spread_last60.reindex(time_ids).fillna(0) /
        spread_first60.reindex(time_ids).fillna(EPS).clip(lower=EPS)
    ).rename("spread_change_ratio")

    spread_rv_corr_vals = vec_spread_rv_corr(
        df_obs["bid_ask_spread"].values,
        df_obs["abs_ret"].values,
        tid_mapped, n_tids
    )
    spread_rv_corr = pd.Series(spread_rv_corr_vals,
                               index=time_ids, name="spread_rv_corr")

    # ── I. Volume features ────────────────────────────────────────────
    vol_mean     = g_obs["total_volume"].mean().rename("vol_mean")
    vol_std      = g_obs["total_volume"].std().rename("vol_std")
    vol_sum      = g_obs["total_volume"].sum().rename("vol_sum")
    vol_max      = g_obs["total_volume"].max().rename("vol_max")
    vol_cv       = (vol_std / vol_mean.clip(lower=EPS)).rename("vol_cv")
    log_vol_mean = g_obs["log_volume"].mean().rename("log_vol_mean")
    log_vol_std  = g_obs["log_volume"].std().rename("log_vol_std")
    vol_last60   = df_obs[df_obs["seconds_in_bucket"] >= 420].groupby(
                   "time_id")["total_volume"].mean().rename("vol_last60")
    vol_ratio    = (vol_last60.reindex(time_ids).fillna(0) /
                    vol_mean.clip(lower=EPS)).rename("vol_ratio")
    vol_skew     = g_obs["total_volume"].skew().rename("vol_skew")
    ofi          = g_obs["signed_vol"].sum().rename("ofi")

    # ── J. Volume-weighted ────────────────────────────────────────────
    tot_vol   = g_obs["total_volume"].sum().clip(lower=EPS)
    vwap_ret  = g_obs["vw_ret"].sum().div(tot_vol).rename("vwap_ret")
    vw_spread = g_obs["vw_spread"].sum().div(tot_vol).rename("vw_spread")

    # ── M. Realized semivariance (vectorised) ─────────────────────────
    semi_df       = vec_semi(df_obs["log_ret"].values, tid_mapped, n_tids)
    semi_df.index = time_ids

    # ── Per 120s bucket RV ────────────────────────────────────────────
    df_obs["bucket_120"] = (df_obs["seconds_in_bucket"] // BUCKET_SIZE).clip(
                            upper=N_BUCKETS - 1).astype(int)
    bucket_frames = []
    for b in range(N_BUCKETS):
        bkt = (df_obs[df_obs["bucket_120"] == b]
               .groupby("time_id")["sq_ret"]
               .sum().apply(np.sqrt)
               .rename(f"past_rv_bkt{b}"))
        bucket_frames.append(bkt)
    bucket_rv_df = pd.concat(bucket_frames, axis=1).reindex(time_ids).fillna(0)
    for b in range(N_BUCKETS):
        bucket_rv_df[f"past_log_rv_bkt{b}"] = safe_log(
            bucket_rv_df[f"past_rv_bkt{b}"].clip(lower=RV_FLOOR).values
        )
    bucket_rv_df["rv_trend_ratio"] = (
        bucket_rv_df[f"past_rv_bkt{N_BUCKETS-1}"] /
        bucket_rv_df["past_rv_bkt0"].clip(lower=EPS)
    )

    # ── Combine all features ──────────────────────────────────────────
    features = pd.concat([
        # C
        rv_full, rv_last60, rv_last120, rv_last240, rv_first_h, rv_second_h,
        log_rv_full, log_rv_last60, log_rv_last120, log_rv_last240,
        log_rv_first_h, log_rv_second_h,
        # D
        rv_ratio_60, rv_ratio_120, rv_ratio_240, rv_ratio_halves,
        log_rv_ratio_60, log_rv_ratio_120, rv_accel,
        # E
        bpv_df,
        # F
        ret_mean, ret_std, ret_skew, ret_kurt,
        ret_min, ret_max, ret_range, ret_iqr,
        abs_ret_mean, abs_ret_max, abs_ret_p95,
        # G
        ret_ac1, abs_ret_ac1,
        # H
        spread_mean, spread_std, spread_max, spread_min, spread_range,
        spread_cv, log_spread_mean, log_spread_std,
        spread_last60, spread_ratio, spread_skew,
        spread_trend, spread_change_ratio, spread_rv_corr,
        # I
        vol_mean, vol_std, vol_sum, vol_max, vol_cv,
        log_vol_mean, log_vol_std, vol_last60, vol_ratio, vol_skew, ofi,
        # J
        vwap_ret, vw_spread,
        # M
        semi_df,
    ], axis=1)

    features.index.name = "time_id"
    features = features.reset_index()
    features = features.merge(interval_rv.reset_index(), on="time_id", how="left")
    features = features.merge(bucket_rv_df.reset_index(), on="time_id", how="left")
    features["stock_id"] = stock_id
    features = features.replace([np.inf, -np.inf], np.nan).fillna(0)
    return features


# ── Build features across all stocks in a fold split ─────────────────

def build_features(raw_df: pd.DataFrame, label: str) -> pd.DataFrame:
    available_gb = psutil.virtual_memory().available / 1e9
    print(f"  RAM before {label}: {available_gb:.2f} GB")
    if available_gb < 2.0:
        raise MemoryError(f"Insufficient RAM ({available_gb:.2f} GB) before {label}")

    stocks     = raw_df["stock_id"].unique()
    stock_dfs  = [raw_df[raw_df["stock_id"] == sid] for sid in stocks]

    results = Parallel(n_jobs=4, prefer="threads", verbose=5)(
        delayed(process_one_stock)(sdf) for sdf in stock_dfs
    )

    out = pd.concat([r for r in results if len(r) > 0], ignore_index=True)
    del stock_dfs, results
    gc.collect()
    print(f"  {label}: {out.shape[0]} rows, {out.shape[1]} columns")
    return out


print("Cell 3 complete — vectorised feature engineering functions defined")
print("Nothing runs until cell 4. Do not call build_features manually.")

Cell 3 complete — vectorised feature engineering functions defined
Nothing runs until cell 4. Do not call build_features manually.


In [10]:
# ── Cell 4: Feature engineering runner (run once, saves to disk) ──────

import glob as glob_module
import pyarrow as pa
import pyarrow.parquet as pq

DENORM_DIR    = Path(r"E:\Optiver\individual_book_train_denorm")
CHECKPOINT_DIR = FEATURES_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def get_fold_time_ids(fold_idx: int):
    """Read only time_id column from fold parquets — cheap, ~10MB per file."""
    train_path = FOLD_PATHS[fold_idx][0]
    test_path  = FOLD_PATHS[fold_idx][1]
    train_tids = set(pd.read_parquet(train_path, columns=["time_id"])["time_id"].unique())
    test_tids  = set(pd.read_parquet(test_path,  columns=["time_id"])["time_id"].unique())
    return train_tids, test_tids

def checkpoint_path(fold_idx: int, stock_id: int) -> Path:
    return CHECKPOINT_DIR / f"fold_{fold_idx}_stock_{stock_id}.parquet"

def is_checkpointed(fold_idx: int, stock_id: int) -> bool:
    return checkpoint_path(fold_idx, stock_id).exists()

def parse_stock_id(fname: str) -> int:
    return int(Path(fname).stem.replace("stock_", ""))

def process_and_checkpoint(stock_file: str, fold_idx: int,
                           train_tids: set, test_tids: set) -> tuple:
    """
    Load one stock CSV, split by fold time_ids, run feature engineering,
    save checkpoint. Returns (stock_id, train_df, test_df) or None on skip.
    """
    stock_id = parse_stock_id(stock_file)

    if is_checkpointed(fold_idx, stock_id):
        # Load from checkpoint
        ckpt = pd.read_parquet(checkpoint_path(fold_idx, stock_id))
        train_chunk = ckpt[ckpt["split"] == "train"].drop(columns="split")
        test_chunk  = ckpt[ckpt["split"] == "test"].drop(columns="split")
        return stock_id, train_chunk, test_chunk

    # Load stock CSV
    df = pd.read_csv(stock_file)
    df["stock_id"] = stock_id

    # Split by fold time_ids
    train_raw = df[df["time_id"].isin(train_tids)]
    test_raw  = df[df["time_id"].isin(test_tids)]

    if train_raw.empty or test_raw.empty:
        return stock_id, pd.DataFrame(), pd.DataFrame()

    # Feature engineering
    train_feat = process_one_stock(train_raw)
    test_feat  = process_one_stock(test_raw)

    if train_feat.empty or test_feat.empty:
        return stock_id, pd.DataFrame(), pd.DataFrame()

    # Save checkpoint
    train_feat["split"] = "train"
    test_feat["split"]  = "test"
    ckpt_df = pd.concat([train_feat, test_feat], ignore_index=True)
    ckpt_df.to_parquet(checkpoint_path(fold_idx, stock_id), index=False)
    train_feat = train_feat.drop(columns="split")
    test_feat  = test_feat.drop(columns="split")

    del df, train_raw, test_raw
    gc.collect()

    return stock_id, train_feat, test_feat


# ── Main loop ─────────────────────────────────────────────────────────
stock_files = sorted(glob_module.glob(str(DENORM_DIR / "stock_*.csv")))
print(f"Found {len(stock_files)} stock files")

for fold_idx in range(5):
    # Check if fold output already exists — skip entirely if so
    train_out = FEATURES_DIR / f"features_fold_{fold_idx}_train.parquet"
    test_out  = FEATURES_DIR / f"features_fold_{fold_idx}_test.parquet"

    if train_out.exists() and test_out.exists():
        print(f"\nFold {fold_idx} already complete — skipping")
        continue

    available_gb = psutil.virtual_memory().available / 1e9
    print(f"\n── Fold {fold_idx} | RAM available: {available_gb:.2f} GB ──")
    if available_gb < 1.0:
        raise MemoryError(f"Only {available_gb:.2f} GB available before fold {fold_idx} — free memory and rerun")

    # Load time_id splits cheaply
    print(f"  Loading time_id splits...")
    train_tids, test_tids = get_fold_time_ids(fold_idx)
    print(f"  Train time_ids: {len(train_tids)} | Test time_ids: {len(test_tids)}")

    # Process stocks with threading
    results = Parallel(n_jobs=4, prefer="threads", verbose=5)(
        delayed(process_and_checkpoint)(sf, fold_idx, train_tids, test_tids)
        for sf in stock_files
    )

    # Aggregate into fold-level parquets
    print(f"  Aggregating fold {fold_idx} outputs...")
    train_frames = [r[1] for r in results if r is not None and len(r[1]) > 0]
    test_frames  = [r[2] for r in results if r is not None and len(r[2]) > 0]

    train_df = pd.concat(train_frames, ignore_index=True)
    test_df  = pd.concat(test_frames,  ignore_index=True)

    train_df.to_parquet(train_out, index=False)
    test_df.to_parquet(test_out,   index=False)

    print(f"  Fold {fold_idx} saved — train: {train_df.shape}, test: {test_df.shape}")

    del train_frames, test_frames, train_df, test_df, results
    gc.collect()

print("\n✓ All folds complete — features saved to outputs/features/")

Found 112 stock files

── Fold 0 | RAM available: 1.24 GB ──
  Loading time_id splits...
  Train time_ids: 3064 | Test time_ids: 766


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:  4.3min
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed: 21.6min
[Parallel(n_jobs=4)]: Done 112 out of 112 | elapsed: 37.7min finished


  Aggregating fold 0 outputs...
  Fold 0 saved — train: (343146, 122), test: (85786, 122)

── Fold 1 | RAM available: 3.36 GB ──
  Loading time_id splits...
  Train time_ids: 3064 | Test time_ids: 766


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:  4.2min
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed: 21.2min
[Parallel(n_jobs=4)]: Done 112 out of 112 | elapsed: 36.8min finished


  Aggregating fold 1 outputs...
  Fold 1 saved — train: (343142, 122), test: (85790, 122)

── Fold 2 | RAM available: 3.25 GB ──
  Loading time_id splits...
  Train time_ids: 3064 | Test time_ids: 766


...

  Aggregating fold 2 outputs...
  Fold 2 saved — train: (343143, 122), test: (85789, 122)

── Fold 3 | RAM available: 3.36 GB ──
  Loading time_id splits...
  Train time_ids: 3064 | Test time_ids: 766


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:  4.3min
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed: 22.4min
[Parallel(n_jobs=4)]: Done 112 out of 112 | elapsed: 38.3min finished


  Aggregating fold 3 outputs...
  Fold 3 saved — train: (343145, 122), test: (85787, 122)

── Fold 4 | RAM available: 3.35 GB ──
  Loading time_id splits...
  Train time_ids: 3064 | Test time_ids: 766


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:  4.0min
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed: 20.6min
[Parallel(n_jobs=4)]: Done 112 out of 112 | elapsed: 36.1min finished


  Aggregating fold 4 outputs...
  Fold 4 saved — train: (343152, 122), test: (85780, 122)

✓ All folds complete — features saved to outputs/features/


In [9]:
# ── Cell 4b: Runtime definitions (rerun after every kernel restart) ────

import glob as glob_module

# Stock files
DENORM_DIR  = Path(r"E:\Optiver\individual_book_train_denorm")
stock_files = sorted(glob_module.glob(str(DENORM_DIR / "stock_*.csv")))
print(f"Found {len(stock_files)} stock files")

# Fold time_id loader
def get_fold_time_ids(fold_idx: int):
    train_path = FOLD_PATHS[fold_idx][0]
    test_path  = FOLD_PATHS[fold_idx][1]
    train_tids = set(pd.read_parquet(train_path, columns=["time_id"])["time_id"].unique())
    test_tids  = set(pd.read_parquet(test_path,  columns=["time_id"])["time_id"].unique())
    return train_tids, test_tids

# Stock id parser
def parse_stock_id(fname: str) -> int:
    return int(Path(fname).stem.replace("stock_", ""))

print("Cell 4b complete — runtime definitions loaded")

Found 112 stock files
Cell 4b complete — runtime definitions loaded


In [13]:
# ── Cell 5: Feature loader ─────────────────────────────────────────────

def load_fold_features(fold_idx: int) -> tuple:
    """
    Load pre-engineered train and test feature parquets for a given fold.
    Called lazily inside the training loop — one fold at a time.
    """
    train_path = FEATURES_DIR / f"features_fold_{fold_idx}_train.parquet"
    test_path  = FEATURES_DIR / f"features_fold_{fold_idx}_test.parquet"

    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(
            f"Features for fold {fold_idx} not found. "
            f"Has cell 4 completed successfully?"
        )

    available_gb = psutil.virtual_memory().available / 1e9
    print(f"  RAM before loading fold {fold_idx}: {available_gb:.2f} GB")
    if available_gb < 1.0:
        raise MemoryError(
            f"Only {available_gb:.2f} GB available before loading fold {fold_idx}"
        )

    train_df = pd.read_parquet(train_path)
    test_df  = pd.read_parquet(test_path)

    # Attach cluster labels
    train_df["cluster_id"] = train_df["stock_id"].map(STOCK_CLUSTER_MAP)
    test_df["cluster_id"]  = test_df["stock_id"].map(STOCK_CLUSTER_MAP)

    print(f"  Fold {fold_idx} loaded — "
          f"train: {train_df.shape}, test: {test_df.shape}")
    print(f"  Clusters in train: "
          f"{ {c: (train_df['cluster_id']==c).sum() for c in sorted(STOCK_CLUSTER_MAP.values())} }")

    return train_df, test_df


print("Cell 5 complete — load_fold_features defined, nothing loaded yet")

Cell 5 complete — load_fold_features defined, nothing loaded yet


In [16]:
# ── Cell 6: Per-stock HAR (OOF train + test predictions) ──────────────
from sklearn.model_selection import GroupKFold

HAR_DIR = Path(r"E:\Optiver\outputs\har_predictions")
HAR_DIR.mkdir(parents=True, exist_ok=True)

N_INNER_FOLDS = 5

def compute_rv(group, sec_min, sec_max):
    """Compute RV over a seconds window for a group."""
    mask = (group["seconds_in_bucket"] >= sec_min) & \
           (group["seconds_in_bucket"] < sec_max)
    sub  = group[mask]
    if sub.empty:
        return RV_FLOOR
    log_ret = np.diff(np.log(np.maximum(sub["wap"].values, EPS)))
    return float(max(np.sqrt(np.sum(log_ret**2)), RV_FLOOR))

def fit_har_ols(X_train, y_train):
    """Fit HAR via OLS, return beta coefficients."""
    try:
        beta = np.linalg.lstsq(X_train, y_train, rcond=None)[0]
        return beta
    except np.linalg.LinAlgError:
        return None

def make_X(df):
    """Build HAR design matrix from rv dataframe."""
    return np.column_stack([
        np.ones(len(df)),
        df["log_rv_in"].values,
        df["log_rv_last_window"].values,
        df["log_rv_ratio"].values,
    ])

def fit_predict_har_stock(stock_file: str,
                          fold_idx: int,
                          train_tids: set,
                          test_tids: set) -> tuple:
    """
    For one stock and one fold:
    - Compute rv_in, rv_last_window, rv_fut per time_id
    - Generate OOF HAR residuals on train time_ids via inner 5-fold CV
    - Fit HAR on all train time_ids, predict on test time_ids
    - Return (train_oof_df, test_df)
    """
    stock_id = parse_stock_id(stock_file)
    ckpt_train = HAR_DIR / f"har_fold_{fold_idx}_stock_{stock_id}_train.parquet"
    ckpt_test  = HAR_DIR / f"har_fold_{fold_idx}_stock_{stock_id}_test.parquet"

    if ckpt_train.exists() and ckpt_test.exists():
        return pd.read_parquet(ckpt_train), pd.read_parquet(ckpt_test)

    df = pd.read_csv(stock_file)
    df["stock_id"] = stock_id

    # ── Compute RV components per time_id ─────────────────────────────
    records = []
    for tid, group in df.groupby("time_id"):
        rv_in          = compute_rv(group, 0,   480)
        rv_last_window = compute_rv(group, 360, 480)
        rv_fut         = compute_rv(group, 480, 600)
        records.append({
            "time_id":            tid,
            "stock_id":           stock_id,
            "rv_in":              rv_in,
            "rv_last_window":     rv_last_window,
            "rv_fut":             rv_fut,
            "log_rv_in":          np.log(max(rv_in,          RV_FLOOR)),
            "log_rv_last_window": np.log(max(rv_last_window, RV_FLOOR)),
            "log_rv_ratio":       np.log(max(rv_last_window, RV_FLOOR)) -
                                  np.log(max(rv_in,          RV_FLOOR)),
            "log_rv_fut":         np.log(max(rv_fut,         RV_FLOOR)),
        })

    rv_df = pd.DataFrame(records)

    # ── Split ─────────────────────────────────────────────────────────
    train_rv = rv_df[rv_df["time_id"].isin(train_tids)].copy().reset_index(drop=True)
    test_rv  = rv_df[rv_df["time_id"].isin(test_tids)].copy().reset_index(drop=True)

    if len(train_rv) < N_INNER_FOLDS * 2 or test_rv.empty:
        return pd.DataFrame(), pd.DataFrame()

    # ── OOF HAR predictions on train time_ids ─────────────────────────
    train_rv["har_pred_log_rv"] = np.nan
    train_rv["har_residual"]    = np.nan

    gkf = GroupKFold(n_splits=N_INNER_FOLDS)
    for inner_train_idx, inner_val_idx in gkf.split(
        train_rv, groups=train_rv["time_id"]
    ):
        X_inner_train = make_X(train_rv.iloc[inner_train_idx])
        y_inner_train = train_rv.iloc[inner_train_idx]["log_rv_fut"].values
        beta = fit_har_ols(X_inner_train, y_inner_train)
        if beta is None:
            continue
        X_inner_val = make_X(train_rv.iloc[inner_val_idx])
        preds = X_inner_val @ beta
        train_rv.loc[inner_val_idx, "har_pred_log_rv"] = preds
        train_rv.loc[inner_val_idx, "har_residual"]    = (
            train_rv.iloc[inner_val_idx]["log_rv_fut"].values - preds
        )

    train_rv["har_pred_rv"] = np.exp(train_rv["har_pred_log_rv"].fillna(0))
    train_rv["fold"]        = fold_idx
    train_rv = train_rv.dropna(subset=["har_residual"])

    # ── Fit on all train, predict on test ─────────────────────────────
    X_train_full = make_X(train_rv)
    y_train_full = train_rv["log_rv_fut"].values
    beta_full    = fit_har_ols(X_train_full, y_train_full)

    if beta_full is None:
        return pd.DataFrame(), pd.DataFrame()

    X_test = make_X(test_rv)
    test_rv["har_pred_log_rv"] = X_test @ beta_full
    test_rv["har_residual"]    = test_rv["log_rv_fut"] - test_rv["har_pred_log_rv"]
    test_rv["har_pred_rv"]     = np.exp(test_rv["har_pred_log_rv"])
    test_rv["fold"]            = fold_idx

    # ── Select output columns ─────────────────────────────────────────
    out_cols = ["time_id", "stock_id", "fold",
                "log_rv_fut", "har_pred_log_rv",
                "har_pred_rv", "har_residual"]

    train_out = train_rv[out_cols].copy()
    test_out  = test_rv[out_cols].copy()

    train_out.to_parquet(ckpt_train, index=False)
    test_out.to_parquet(ckpt_test,   index=False)

    del df, rv_df, train_rv, test_rv
    gc.collect()

    return train_out, test_out


def build_har_predictions(fold_idx: int,
                          train_tids: set,
                          test_tids: set) -> tuple:
    """Run per-stock HAR for all stocks in a fold."""
    train_out_path = HAR_DIR / f"har_fold_{fold_idx}_train.parquet"
    test_out_path  = HAR_DIR / f"har_fold_{fold_idx}_test.parquet"

    if train_out_path.exists() and test_out_path.exists():
        print(f"  Fold {fold_idx} HAR already complete — loading from disk")
        return (pd.read_parquet(train_out_path),
                pd.read_parquet(test_out_path))

    results = Parallel(n_jobs=4, prefer="threads", verbose=5)(
        delayed(fit_predict_har_stock)(sf, fold_idx, train_tids, test_tids)
        for sf in stock_files
    )

    train_frames = [r[0] for r in results if len(r[0]) > 0]
    test_frames  = [r[1] for r in results if len(r[1]) > 0]

    train_df = pd.concat(train_frames, ignore_index=True)
    test_df  = pd.concat(test_frames,  ignore_index=True)

    train_df.to_parquet(train_out_path, index=False)
    test_df.to_parquet(test_out_path,   index=False)

    print(f"  Fold {fold_idx} HAR saved — "
          f"train: {train_df.shape}, test: {test_df.shape}")

    del results, train_frames, test_frames
    gc.collect()
    return train_df, test_df


# ── Run HAR for all folds ─────────────────────────────────────────────
print("Running per-stock HAR across all 5 folds...")
har_train_folds = []
har_test_folds  = []

for fold_idx in range(5):
    print(f"\n── Fold {fold_idx} ──")
    train_tids, test_tids = get_fold_time_ids(fold_idx)
    train_har, test_har   = build_har_predictions(fold_idx, train_tids, test_tids)
    har_train_folds.append(train_har)
    har_test_folds.append(test_har)

    print(f"  Train OOF residual — "
          f"mean: {train_har['har_residual'].mean():.6f}, "
          f"std:  {train_har['har_residual'].std():.4f}")
    print(f"  Test residual — "
          f"mean: {test_har['har_residual'].mean():.4f}, "
          f"std:  {test_har['har_residual'].std():.4f}")

har_train_combined = pd.concat(har_train_folds, ignore_index=True)
har_test_combined  = pd.concat(har_test_folds,  ignore_index=True)

print(f"\n✓ HAR complete")
print(f"  Train OOF: {har_train_combined.shape[0]} rows, "
      f"std: {har_train_combined['har_residual'].std():.4f}")
print(f"  Test:      {har_test_combined.shape[0]} rows, "
      f"std: {har_test_combined['har_residual'].std():.4f}")

Running per-stock HAR across all 5 folds...

── Fold 0 ──
  Fold 0 HAR already complete — loading from disk
  Train OOF residual — mean: -0.000002, std:  0.3205
  Test residual — mean: -0.0014, std:  0.3247

── Fold 1 ──
  Fold 1 HAR already complete — loading from disk
  Train OOF residual — mean: 0.000050, std:  0.3206
  Test residual — mean: 0.0036, std:  0.3244

── Fold 2 ──
  Fold 2 HAR already complete — loading from disk
  Train OOF residual — mean: -0.000032, std:  0.3208
  Test residual — mean: 0.0013, std:  0.3235

── Fold 3 ──
  Fold 3 HAR already complete — loading from disk
  Train OOF residual — mean: -0.000006, std:  0.3219
  Test residual — mean: 0.0024, std:  0.3192

── Fold 4 ──
  Fold 4 HAR already complete — loading from disk
  Train OOF residual — mean: 0.000017, std:  0.3202
  Test residual — mean: 0.0027, std:  0.3258

✓ HAR complete
  Train OOF: 1715728 rows, std: 0.3208
  Test:      428932 rows, std: 0.3235


In [17]:
# ── Cell 7: Target computation and feature-HAR merge ──────────────────
import pyarrow.parquet as pq

META_COLS  = ["time_id", "stock_id", "cluster_id", "fold"]
TARGET_COL = "har_residual"
ANCHOR_COL = "past_log_rv_bkt3"

# ── Identify feature columns from parquet metadata (zero RAM) ──────────
schema           = pq.read_schema(FEATURES_DIR / "features_fold_0_train.parquet")
raw_feature_cols = schema.names

EXCLUDE_COLS     = set(META_COLS + [TARGET_COL, "sample_weight",
                                    "log_rv_fut", "har_pred_log_rv",
                                    "har_pred_rv"])
ALL_FEATURE_COLS = [c for c in raw_feature_cols if c not in EXCLUDE_COLS]

print(f"Total features available: {len(ALL_FEATURE_COLS)}")
print(f"Target : {TARGET_COL}")
print(f"Anchor : {ANCHOR_COL} "
      f"({'present' if ANCHOR_COL in ALL_FEATURE_COLS else '⚠️ MISSING'})")

# ── Lightweight fold verification ──────────────────────────────────────
def verify_fold_merge(fold_idx: int):
    train_feat = pd.read_parquet(
        FEATURES_DIR / f"features_fold_{fold_idx}_train.parquet",
        columns=["time_id", "stock_id"]
    )
    test_feat = pd.read_parquet(
        FEATURES_DIR / f"features_fold_{fold_idx}_test.parquet",
        columns=["time_id", "stock_id"]
    )
    train_feat["cluster_id"] = train_feat["stock_id"].map(STOCK_CLUSTER_MAP)
    test_feat["cluster_id"]  = test_feat["stock_id"].map(STOCK_CLUSTER_MAP)

    train_har = pd.read_parquet(
        HAR_DIR / f"har_fold_{fold_idx}_train.parquet",
        columns=["time_id", "stock_id", "har_residual"]
    )
    test_har = pd.read_parquet(
        HAR_DIR / f"har_fold_{fold_idx}_test.parquet",
        columns=["time_id", "stock_id", "har_residual"]
    )

    train_merged = train_feat.merge(train_har, on=["time_id", "stock_id"], how="inner")
    test_merged  = test_feat.merge(test_har,   on=["time_id", "stock_id"], how="inner")

    train_loss = len(train_feat) - len(train_merged)
    test_loss  = len(test_feat)  - len(test_merged)

    print(f"\n── Fold {fold_idx} ──")
    print(f"  Train: {len(train_feat)} → {len(train_merged)} "
          f"(lost {train_loss}) {'✓' if train_loss == 0 else '⚠️'}")
    print(f"  Test:  {len(test_feat)} → {len(test_merged)} "
          f"(lost {test_loss}) {'✓' if test_loss == 0 else '⚠️'}")
    print(f"  Stocks train: {train_merged['stock_id'].nunique()} | "
          f"test: {test_merged['stock_id'].nunique()}")
    print(f"  Cluster distribution (train):")
    for c in sorted(train_merged["cluster_id"].unique()):
        n_stocks = train_merged[train_merged["cluster_id"] == c]["stock_id"].nunique()
        n_rows   = (train_merged["cluster_id"] == c).sum()
        print(f"    Cluster {c}: {n_stocks} stocks, {n_rows} rows")

    # time_id overlap check
    train_tids = set(train_merged["time_id"].unique())
    test_tids  = set(test_merged["time_id"].unique())
    overlap    = train_tids & test_tids
    print(f"  time_id overlap between train/test: "
          f"{len(overlap)} {'✓' if len(overlap) == 0 else '⚠️ LEAKAGE'}")

    del train_feat, test_feat, train_har, test_har, train_merged, test_merged
    gc.collect()


print("\nVerifying all 5 folds...")
for fold_idx in range(5):
    verify_fold_merge(fold_idx)

# ── prepare_fold_data (called lazily inside training loop) ────────────
def prepare_fold_data(fold_idx: int) -> tuple:
    """
    Load pre-engineered features and HAR predictions for a fold,
    merge them, compute sample weights, return train and test dataframes.
    """
    available_gb = psutil.virtual_memory().available / 1e9
    print(f"  RAM before loading fold {fold_idx}: {available_gb:.2f} GB")
    if available_gb < 1.0:
        raise MemoryError(
            f"Only {available_gb:.2f} GB available — need 1.0GB minimum"
        )

    # ── Load features ─────────────────────────────────────────────────
    train_feat, test_feat = load_fold_features(fold_idx)

    # ── Load HAR predictions ──────────────────────────────────────────
    train_har = pd.read_parquet(HAR_DIR / f"har_fold_{fold_idx}_train.parquet")
    test_har  = pd.read_parquet(HAR_DIR / f"har_fold_{fold_idx}_test.parquet")

    # ── Merge ─────────────────────────────────────────────────────────
    train_df = train_feat.merge(train_har, on=["time_id", "stock_id"], how="inner")
    test_df  = test_feat.merge(test_har,   on=["time_id", "stock_id"], how="inner")

    # ── Row loss check ────────────────────────────────────────────────
    train_loss = len(train_feat) - len(train_df)
    test_loss  = len(test_feat)  - len(test_df)
    if train_loss > 0 or test_loss > 0:
        print(f"  ⚠️  Fold {fold_idx}: lost {train_loss} train rows, "
              f"{test_loss} test rows on merge")
    else:
        print(f"  Fold {fold_idx}: merge clean ✓ — "
              f"train {train_df.shape}, test {test_df.shape}")

    # ── Sample weights (QLIKE proxy: 1/true_rv, capped at 99th pct) ───
    true_rv_train             = np.exp(train_df["log_rv_fut"].values).clip(min=RV_FLOOR)
    raw_weights               = 1.0 / true_rv_train
    weight_cap                = np.percentile(raw_weights, 99)
    train_df["sample_weight"] = np.minimum(raw_weights, weight_cap)

    del train_feat, test_feat, train_har, test_har
    gc.collect()

    return train_df, test_df


print("\nCell 7 complete — all folds verified, prepare_fold_data defined")
print("⚠️  prepare_fold_data is called lazily inside the training loop")

Total features available: 120
Target : har_residual
Anchor : past_log_rv_bkt3 (present)

Verifying all 5 folds...

── Fold 0 ──
  Train: 343146 → 343146 (lost 0) ✓
  Test:  85786 → 85786 (lost 0) ✓
  Stocks train: 112 | test: 112
  Cluster distribution (train):
    Cluster 0: 10 stocks, 30639 rows
    Cluster 1: 90 stocks, 275739 rows
    Cluster 2: 12 stocks, 36768 rows
  time_id overlap between train/test: 0 ✓

── Fold 1 ──
  Train: 343142 → 343142 (lost 0) ✓
  Test:  85790 → 85790 (lost 0) ✓
  Stocks train: 112 | test: 112
  Cluster distribution (train):
    Cluster 0: 10 stocks, 30639 rows
    Cluster 1: 90 stocks, 275735 rows
    Cluster 2: 12 stocks, 36768 rows
  time_id overlap between train/test: 0 ✓

── Fold 2 ──
  Train: 343143 → 343143 (lost 0) ✓
  Test:  85789 → 85789 (lost 0) ✓
  Stocks train: 112 | test: 112
  Cluster distribution (train):
    Cluster 0: 10 stocks, 30639 rows
    Cluster 1: 90 stocks, 275736 rows
    Cluster 2: 12 stocks, 36768 rows
  time_id overlap betw

In [18]:
MI_THRESHOLD   = 0.01
CORR_THRESHOLD = 0.70
SELECTION_SEED = SEED
SELECTED_FEATURES_PATH = Path(r"E:\Optiver\outputs\selected_features.txt")

# ── Memory check ──────────────────────────────────────────────────────
available_gb = psutil.virtual_memory().available / 1e9
print(f"Available RAM: {available_gb:.2f} GB")
if available_gb < 1.0:
    raise MemoryError(
        f"Only {available_gb:.2f} GB available — need 1.0GB before feature selection"
    )

# ── Load fold 0 train ─────────────────────────────────────────────────
print("Loading fold 0 train for feature selection...")
train_df, _ = prepare_fold_data(0)

X = train_df[ALL_FEATURE_COLS].copy()
y = train_df[TARGET_COL].copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median())
y = y.fillna(y.median())

print(f"Feature matrix: {X.shape}")
print(f"Target: {y.describe().to_dict()}")

del train_df
gc.collect()

# ── Stage 1: Mutual Information ───────────────────────────────────────
print(f"\n═══ Stage 1: Mutual Information (threshold={MI_THRESHOLD}) ═══")
mi        = mutual_info_regression(X, y, random_state=SELECTION_SEED, n_neighbors=5)
mi_series = pd.Series(mi, index=ALL_FEATURE_COLS).sort_values(ascending=False)

survivors_mi = mi_series[mi_series >= MI_THRESHOLD].index.tolist()
dropped_mi   = mi_series[mi_series <  MI_THRESHOLD].index.tolist()

print(f"  Survivors: {len(survivors_mi)} | Dropped: {len(dropped_mi)}")
print(f"  Top 15 by MI:")
print(mi_series.head(15).to_string())
if dropped_mi:
    print(f"  Dropped: {dropped_mi[:10]}{'...' if len(dropped_mi) > 10 else ''}")

X_mi = X[survivors_mi].copy()
del X
gc.collect()

# ── Stage 2: LightGBM + SHAP ──────────────────────────────────────────
print(f"\n═══ Stage 2: LightGBM + SHAP ═══")
X_tr, X_val, y_tr, y_val = train_test_split(
    X_mi, y, test_size=0.2, random_state=SELECTION_SEED
)

lgb_params = {
    "objective":         "regression",
    "metric":            "rmse",
    "learning_rate":     0.05,
    "num_leaves":        63,
    "max_depth":         8,
    "min_child_samples": 20,
    "subsample":         0.8,
    "colsample_bytree":  0.8,
    "reg_alpha":         0.1,
    "reg_lambda":        1.0,
    "verbose":           -1,
    "seed":              SELECTION_SEED,
}

dtrain = lgb.Dataset(X_tr, y_tr)
dval   = lgb.Dataset(X_val, y_val, reference=dtrain)

model = lgb.train(
    lgb_params, dtrain,
    num_boost_round=2000,
    valid_sets=[dtrain, dval],
    callbacks=[
        lgb.early_stopping(50, verbose=True),
        lgb.log_evaluation(100),
    ],
)

gain_importance = pd.Series(
    model.feature_importance(importance_type="gain"),
    index=survivors_mi,
).sort_values(ascending=False)
print(f"\n  Top 15 by gain:")
print(gain_importance.head(15).to_string())

print("\n  Computing SHAP values...")
explainer       = shap.TreeExplainer(model)
shap_values     = explainer.shap_values(X_val)
shap_importance = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=survivors_mi,
).sort_values(ascending=False)
print(f"  Top 15 by SHAP:")
print(shap_importance.head(15).to_string())

del X_tr, X_val, y_tr, y_val, dtrain, dval, shap_values
gc.collect()

# ── Stage 3: Correlation clustering ───────────────────────────────────
print(f"\n═══ Stage 3: Correlation Clustering (threshold={CORR_THRESHOLD}) ═══")

corr_matrix = np.abs(X_mi.corr().values)
np.fill_diagonal(corr_matrix, 1.0)
corr_matrix = np.clip(corr_matrix, 0, 1)
distance    = 1.0 - corr_matrix
distance    = (distance + distance.T) / 2
np.fill_diagonal(distance, 0.0)
distance    = np.clip(distance, 0, None)

condensed = squareform(distance, checks=False)
Z         = linkage(condensed, method="average")
clusters  = fcluster(Z, t=1.0 - CORR_THRESHOLD, criterion="distance")

cluster_map = pd.DataFrame({
    "feature": survivors_mi,
    "cluster": clusters,
    "shap":    [shap_importance.get(f, 0.0) for f in survivors_mi],
})

selected_decorr = []
for cid, grp in cluster_map.groupby("cluster"):
    best = grp.sort_values("shap", ascending=False).iloc[0]
    selected_decorr.append(best["feature"])

print(f"  {cluster_map['cluster'].nunique()} correlation clusters "
      f"from {len(survivors_mi)} features")
print(f"  Selected {len(selected_decorr)} decorrelated features")
for cid, grp in cluster_map.groupby("cluster"):
    if len(grp) > 1:
        feats = grp.sort_values("shap", ascending=False)["feature"].tolist()
        print(f"  Cluster {cid}: kept '{feats[0]}', dropped {feats[1:]}")

X_decorr = X_mi[selected_decorr].copy()
del X_mi
gc.collect()

# ── Stage 4: Permutation importance ───────────────────────────────────
print(f"\n═══ Stage 4: Permutation Importance ═══")

X_tr_f, X_val_f, y_tr_f, y_val_f = train_test_split(
    X_decorr, y, test_size=0.2, random_state=SELECTION_SEED
)

dtrain_f = lgb.Dataset(X_tr_f, y_tr_f)
dval_f   = lgb.Dataset(X_val_f, y_val_f, reference=dtrain_f)
model_f  = lgb.train(
    lgb_params, dtrain_f,
    num_boost_round=2000,
    valid_sets=[dtrain_f, dval_f],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100),
    ],
)

class _LGBWrapper:
    def __init__(self, m): self.m = m
    def fit(self, X, y):   return self
    def predict(self, X):  return self.m.predict(X)

perm = permutation_importance(
    _LGBWrapper(model_f), X_val_f, y_val_f,
    n_repeats=10, random_state=SELECTION_SEED,
    scoring="neg_mean_squared_error",
)
perm_importance = pd.Series(
    perm.importances_mean, index=selected_decorr
).sort_values(ascending=False)

perm_survivors = perm_importance[perm_importance > 0].index.tolist()
neg_perm       = perm_importance[perm_importance <= 0].index.tolist()

print(f"  Survivors: {len(perm_survivors)} | Dropped: {len(neg_perm)}")
print(f"  Top 15 by permutation importance:")
print(perm_importance.head(15).to_string())
if neg_perm:
    print(f"  Dropped (negative perm importance): {neg_perm}")

del X_tr_f, X_val_f, y_tr_f, y_val_f, dtrain_f, dval_f, X_decorr
gc.collect()

# ── Summary ───────────────────────────────────────────────────────────
print(f"\n{'═' * 60}")
print(f"Feature Selection Summary")
print(f"{'═' * 60}")
print(f"  Initial features             : {len(ALL_FEATURE_COLS)}")
print(f"  After MI filter              : {len(survivors_mi)}")
print(f"  After correlation clustering : {len(selected_decorr)}")
print(f"  After permutation importance : {len(perm_survivors)}")
print(f"\n  FINAL FEATURES ({len(perm_survivors)}):")
for i, f in enumerate(perm_survivors):
    shap_v = shap_importance.get(f, 0.0)
    perm_v = perm_importance.get(f, 0.0)
    print(f"    {i+1:>3}. {f:<40s} SHAP={shap_v:.4f}  Perm={perm_v:.6f}")
print(f"{'═' * 60}")

# ── Save selected features ────────────────────────────────────────────
SELECTED_FEATURES = perm_survivors
with open(SELECTED_FEATURES_PATH, "w") as f:
    f.write("\n".join(SELECTED_FEATURES))
print(f"\n✓ Saved {len(SELECTED_FEATURES)} selected features to {SELECTED_FEATURES_PATH}")
print("\nCell 8 complete")

Available RAM: 2.51 GB
Loading fold 0 train for feature selection...
  RAM before loading fold 0: 2.51 GB
  RAM before loading fold 0: 2.51 GB
  Fold 0 loaded — train: (343146, 123), test: (85786, 123)
  Clusters in train: {0: np.int64(30639), 1: np.int64(275739), 2: np.int64(36768)}
  Fold 0: merge clean ✓ — train (343146, 128), test (85786, 128)
Feature matrix: (343146, 120)
Target: {'count': 343146.0, 'mean': -2.373357119774462e-06, 'std': 0.3205043878733748, 'min': -3.8853404253613073, '25%': -0.16521220870316844, '50%': 0.014741030265355004, '75%': 0.18808535550465444, 'max': 2.5551405451617573}

═══ Stage 1: Mutual Information (threshold=0.01) ═══
  Survivors: 41 | Dropped: 79
  Top 15 by MI:
rv_lag_std         0.056311
ret_kurt           0.053417
ret_iqr            0.042563
jump_ratio         0.033652
har_short_long     0.018761
har_med_long       0.018444
log_rv_medium      0.018282
har_short_med      0.018200
rv_trend_slope     0.017194
spread_range       0.017105
spread_mean 

In [19]:

# ── Cell 9: Cross-stock z-scores ──────────────────────────────────────
ZSCORE_DIR = Path(r"E:\Optiver\outputs\zscore_stats")
ZSCORE_DIR.mkdir(parents=True, exist_ok=True)

ZSCORE_FEATURES = SELECTED_FEATURES  # 21 features
ZSCORE_COLS     = [f"{f}_zscore" for f in ZSCORE_FEATURES]

def compute_fold_zscore_stats(fold_idx: int, split: str) -> pd.DataFrame:
    """
    Step 1 — Load only selected feature columns + time_id for a fold split,
    compute per-time_id mean and std for each selected feature.
    Saves to disk and returns stats dataframe.
    """
    stats_path = ZSCORE_DIR / f"zscore_stats_fold_{fold_idx}_{split}.parquet"
    if stats_path.exists():
        print(f"  Fold {fold_idx} {split} zscore stats already exist — loading")
        return pd.read_parquet(stats_path)

    feat_path = FEATURES_DIR / f"features_fold_{fold_idx}_{split}.parquet"
    df = pd.read_parquet(feat_path, columns=["time_id", "stock_id"] + ZSCORE_FEATURES)

    # Compute per-time_id mean and std for each feature
    means = df.groupby("time_id")[ZSCORE_FEATURES].mean()
    stds  = df.groupby("time_id")[ZSCORE_FEATURES].std().clip(lower=EPS)

    means.columns = [f"{c}_mean" for c in means.columns]
    stds.columns  = [f"{c}_std"  for c in stds.columns]

    stats = pd.concat([means, stds], axis=1).reset_index()
    stats.to_parquet(stats_path, index=False)

    print(f"  Fold {fold_idx} {split}: zscore stats computed "
          f"({stats.shape[0]} time_ids, {stats.shape[1]} columns)")

    del df, means, stds
    gc.collect()
    return stats


def apply_zscores_to_fold(fold_idx: int, split: str,
                          stats: pd.DataFrame) -> None:
    """
    Step 2 — Load feature parquet, merge zscore stats,
    compute z-scores, save updated parquet back to disk.
    """
    out_path = FEATURES_DIR / f"features_fold_{fold_idx}_{split}_with_zscores.parquet"
    if out_path.exists():
        print(f"  Fold {fold_idx} {split} zscores already applied — skipping")
        return

    feat_path = FEATURES_DIR / f"features_fold_{fold_idx}_{split}.parquet"
    df = pd.read_parquet(feat_path)

    # Merge stats
    df = df.merge(stats, on="time_id", how="left")

    # Compute z-scores
    for f in ZSCORE_FEATURES:
        mean_col      = f"{f}_mean"
        std_col       = f"{f}_std"
        zscore_col    = f"{f}_zscore"
        df[zscore_col] = (df[f] - df[mean_col]) / df[std_col]

    # Drop intermediate mean/std columns
    stat_cols = ([f"{f}_mean" for f in ZSCORE_FEATURES] +
                 [f"{f}_std"  for f in ZSCORE_FEATURES])
    df = df.drop(columns=stat_cols)

    # Clean
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
    df.to_parquet(out_path, index=False)

    print(f"  Fold {fold_idx} {split}: zscores applied — "
          f"{df.shape[0]} rows, {df.shape[1]} columns")

    del df
    gc.collect()


# ── Run across all folds ──────────────────────────────────────────────
print("Computing cross-stock z-scores across all folds...")

for fold_idx in range(5):
    print(f"\n── Fold {fold_idx} ──")
    available_gb = psutil.virtual_memory().available / 1e9
    print(f"  RAM available: {available_gb:.2f} GB")

    for split in ["train", "test"]:
        stats = compute_fold_zscore_stats(fold_idx, split)
        apply_zscores_to_fold(fold_idx, split, stats)
        del stats
        gc.collect()

# ── Update feature list ───────────────────────────────────────────────
FINAL_FEATURES = SELECTED_FEATURES + ZSCORE_COLS
print(f"\n✓ Z-scores complete")
print(f"  Selected features : {len(SELECTED_FEATURES)}")
print(f"  Z-score features  : {len(ZSCORE_COLS)}")
print(f"  Total features    : {len(FINAL_FEATURES)}")

# Save final feature list
FINAL_FEATURES_PATH = Path(r"E:\Optiver\outputs\final_features.txt")
with open(FINAL_FEATURES_PATH, "w") as f:
    f.write("\n".join(FINAL_FEATURES))
print(f"  Saved to {FINAL_FEATURES_PATH}")

print("\nCell 9 complete")
print("⚠️  LightGBM training (cell 10) reads from features_fold_*_with_zscores.parquet")

Computing cross-stock z-scores across all folds...

── Fold 0 ──
  RAM available: 1.30 GB
  Fold 0 train: zscore stats computed (3064 time_ids, 43 columns)
  Fold 0 train: zscores applied — 343146 rows, 143 columns
  Fold 0 test: zscore stats computed (766 time_ids, 43 columns)
  Fold 0 test: zscores applied — 85786 rows, 143 columns

── Fold 1 ──
  RAM available: 2.69 GB
  Fold 1 train: zscore stats computed (3064 time_ids, 43 columns)
  Fold 1 train: zscores applied — 343142 rows, 143 columns
  Fold 1 test: zscore stats computed (766 time_ids, 43 columns)
  Fold 1 test: zscores applied — 85790 rows, 143 columns

── Fold 2 ──
  RAM available: 3.01 GB
  Fold 2 train: zscore stats computed (3064 time_ids, 43 columns)
  Fold 2 train: zscores applied — 343143 rows, 143 columns
  Fold 2 test: zscore stats computed (766 time_ids, 43 columns)
  Fold 2 test: zscores applied — 85789 rows, 143 columns

── Fold 3 ──
  RAM available: 2.81 GB
  Fold 3 train: zscore stats computed (3064 time_ids, 4

In [23]:
# ── Cell 10: Stock-level LightGBM with Optuna + anchor-aware feval ────
# ⚠️  OVERNIGHT RUN — 4-6 hours expected
# ⚠️  Ensure sleep disabled and E: has 10GB+ free

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

STOCK_LGB_DIR   = Path(r"E:\Optiver\outputs\stock_lgbm")
STOCK_LGB_DIR.mkdir(parents=True, exist_ok=True)

N_OPTUNA_TRIALS = 10
N_INNER_FOLDS   = 5

def rmspe(y_true, y_pred):
    y_pred = np.maximum(y_pred, EPS)
    return float(np.sqrt(np.mean(((y_true - y_pred) / y_true)**2)))

def qlike(y_true, y_pred):
    y_pred = np.maximum(y_pred, EPS)
    return float(np.mean(np.log(y_pred) + y_true / y_pred))

def anchor_rmspe_feval(y_pred, dtrain):
    """
    Anchor-aware RMSPE feval.
    Reconstructs final RV from residual + HAR anchor before computing RMSPE.
    """
    y_residual  = dtrain.get_label()
    anchor      = dtrain.anchor_log_rv
    pred_log_rv = np.clip(y_pred + anchor, -20, 5)
    true_log_rv = y_residual + anchor
    pred_rv     = np.exp(pred_log_rv).clip(min=EPS)
    true_rv     = np.exp(true_log_rv).clip(min=EPS)
    score       = float(np.sqrt(np.mean(((pred_rv - true_rv) / true_rv)**2)))
    return "rmspe", score, False

def load_fold_with_zscores(fold_idx: int, split: str) -> pd.DataFrame:
    """Load feature parquet with z-scores and merge HAR predictions."""
    feat_path = FEATURES_DIR / f"features_fold_{fold_idx}_{split}_with_zscores.parquet"
    df = pd.read_parquet(feat_path)
    df["cluster_id"] = df["stock_id"].map(STOCK_CLUSTER_MAP)

    har_path = HAR_DIR / f"har_fold_{fold_idx}_{split}.parquet"
    har_df   = pd.read_parquet(har_path)
    df = df.merge(har_df, on=["time_id", "stock_id"], how="inner")

    true_rv             = np.exp(df["log_rv_fut"].values).clip(min=RV_FLOOR)
    raw_weights         = 1.0 / true_rv
    weight_cap          = np.percentile(raw_weights, 99)
    df["sample_weight"] = np.minimum(raw_weights, weight_cap)

    return df

def make_optuna_objective(X_train, y_train, w_train,
                          anchor_train, time_ids_train):
    """
    Returns an Optuna objective function that runs inner GroupKFold CV
    with anchor-aware RMSPE as the optimisation target.
    """
    def objective(trial):
        params = {
            "objective":         "regression",
            "metric":            "None",
            "boosting_type":     "gbdt",
            "verbosity":         -1,
            "seed":              SEED,
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "num_leaves":        trial.suggest_int("num_leaves", 15, 200),
            "max_depth":         trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
            "feature_fraction":  trial.suggest_float("feature_fraction", 0.4, 1.0),
            "bagging_fraction":  trial.suggest_float("bagging_fraction", 0.4, 1.0),
            "bagging_freq":      trial.suggest_int("bagging_freq", 1, 7),
            "reg_alpha":         trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }

        gkf    = GroupKFold(n_splits=N_INNER_FOLDS)
        scores = []

        for tr_idx, va_idx in gkf.split(X_train, groups=time_ids_train):
            X_tr, X_va = X_train[tr_idx], X_train[va_idx]
            y_tr, y_va = y_train[tr_idx], y_train[va_idx]
            w_tr       = w_train[tr_idx]
            anc_tr     = anchor_train[tr_idx]
            anc_va     = anchor_train[va_idx]

            d_tr = lgb.Dataset(X_tr, label=y_tr, weight=w_tr,
                               feature_name=FINAL_FEATURES, free_raw_data=False)
            d_va = lgb.Dataset(X_va, label=y_va,
                               feature_name=FINAL_FEATURES,
                               reference=d_tr, free_raw_data=False)
            d_tr.anchor_log_rv = anc_tr
            d_va.anchor_log_rv = anc_va

            m = lgb.train(
                params, d_tr,
                num_boost_round=N_ROUNDS,
                valid_sets=[d_va],
                feval=anchor_rmspe_feval,
                callbacks=[
                    lgb.early_stopping(EARLY_STOPPING, verbose=False),
                    lgb.log_evaluation(-1),
                ],
            )

            # Reconstruct final RV for scoring
            va_pred     = m.predict(X_va)
            pred_log_rv = np.clip(va_pred + anc_va, -20, 5)
            true_log_rv = y_va + anc_va
            pred_rv     = np.exp(pred_log_rv).clip(min=EPS)
            true_rv     = np.exp(true_log_rv).clip(min=EPS)
            scores.append(float(np.sqrt(np.mean(((pred_rv - true_rv) / true_rv)**2))))

            del d_tr, d_va, m
            gc.collect()

        return float(np.mean(scores))
    return objective


def train_stock_lgbm(fold_idx: int) -> dict:
    pred_path  = STOCK_LGB_DIR / f"predictions_fold_{fold_idx}.parquet"
    model_path = STOCK_LGB_DIR / f"model_fold_{fold_idx}.txt"
    params_path = STOCK_LGB_DIR / f"best_params_fold_{fold_idx}.json"

    if pred_path.exists() and model_path.exists():
        print(f"  Fold {fold_idx} already complete — loading from disk")
        return {"predictions": pd.read_parquet(pred_path)}

    available_gb = psutil.virtual_memory().available / 1e9
    print(f"  RAM before fold {fold_idx}: {available_gb:.2f} GB")
    if available_gb < 1.5:
        raise MemoryError(f"Only {available_gb:.2f} GB available")

    # ── Load data ─────────────────────────────────────────────────────
    print(f"  Loading fold {fold_idx} train...")
    train_df = load_fold_with_zscores(fold_idx, "train")
    print(f"  Loading fold {fold_idx} test...")
    test_df  = load_fold_with_zscores(fold_idx, "test")

    X_train      = train_df[FINAL_FEATURES].values.astype(np.float32)
    y_train      = train_df[TARGET_COL].values.astype(np.float32)
    w_train      = train_df["sample_weight"].values.astype(np.float32)
    anchor_train = train_df["har_pred_log_rv"].values.astype(np.float32)
    time_ids_tr  = train_df["time_id"].values
    X_test       = test_df[FINAL_FEATURES].values.astype(np.float32)
    anchor_test  = test_df["har_pred_log_rv"].values.astype(np.float32)

    # ── Optuna hyperparameter search ──────────────────────────────────
    print(f"  Running Optuna ({N_OPTUNA_TRIALS} trials)...")
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(
        make_optuna_objective(X_train, y_train, w_train,
                              anchor_train, time_ids_tr),
        n_trials=N_OPTUNA_TRIALS,
        show_progress_bar=False,
    )
    best_params = study.best_params
    best_params.update({
        "objective":     "regression",
        "metric":        "None",
        "boosting_type": "gbdt",
        "verbosity":     -1,
        "seed":          SEED,
    })
    print(f"  Best inner RMSPE: {study.best_value:.6f}")
    print(f"  Best params: {json.dumps({k: v for k, v in best_params.items() if k not in ['objective','metric','boosting_type','verbosity','seed']}, indent=4)}")

    # Save best params
    with open(params_path, "w") as f:
        json.dump({
            "best_params":       best_params,
            "best_inner_rmspe":  study.best_value,
            "n_optuna_trials":   N_OPTUNA_TRIALS,
        }, f, indent=2, default=str)

    # ── Final fit on full train with early stopping holdout ───────────
    print(f"  Fitting final model on full train...")
    rng    = np.random.default_rng(SEED)
    idx    = rng.permutation(len(X_train))
    n_es   = max(1, int(len(X_train) * 0.1))
    es_idx = idx[:n_es]
    tr_idx = idx[n_es:]

    d_tr = lgb.Dataset(X_train[tr_idx], label=y_train[tr_idx],
                       weight=w_train[tr_idx],
                       feature_name=FINAL_FEATURES, free_raw_data=False)
    d_es = lgb.Dataset(X_train[es_idx], label=y_train[es_idx],
                       feature_name=FINAL_FEATURES,
                       reference=d_tr, free_raw_data=False)
    d_tr.anchor_log_rv = anchor_train[tr_idx]
    d_es.anchor_log_rv = anchor_train[es_idx]

    model_final = lgb.train(
        best_params, d_tr,
        num_boost_round=N_ROUNDS,
        valid_sets=[d_es],
        feval=anchor_rmspe_feval,
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING, verbose=False),
            lgb.log_evaluation(200),
        ],
    )
    print(f"  Best iteration: {model_final.best_iteration}")
    model_final.save_model(str(model_path))

    # ── Predict on test ───────────────────────────────────────────────
    lgb_residual_pred = model_final.predict(X_test)
    har_pred_log_rv   = test_df["har_pred_log_rv"].values
    final_pred_log_rv = np.clip(har_pred_log_rv + lgb_residual_pred, -20, 5)
    actual_log_rv     = test_df["log_rv_fut"].values

    actual_rv     = np.exp(actual_log_rv).clip(min=EPS)
    final_pred_rv = np.exp(final_pred_log_rv).clip(min=EPS)
    har_pred_rv   = np.exp(har_pred_log_rv).clip(min=EPS)

    # ── Metrics ───────────────────────────────────────────────────────
    rmspe_hybrid = rmspe(actual_rv, final_pred_rv)
    qlike_hybrid = qlike(actual_rv, final_pred_rv)
    rmspe_har    = rmspe(actual_rv, har_pred_rv)
    qlike_har    = qlike(actual_rv, har_pred_rv)

    print(f"\n  Fold {fold_idx} Results:")
    print(f"    HAR          — RMSPE: {rmspe_har:.6f}, QLIKE: {qlike_har:.6f}")
    print(f"    HAR+LightGBM — RMSPE: {rmspe_hybrid:.6f}, QLIKE: {qlike_hybrid:.6f}")
    print(f"    RMSPE improvement: {(rmspe_har - rmspe_hybrid)/rmspe_har*100:.2f}%")
    print(f"    QLIKE improvement: {(qlike_har - qlike_hybrid)/qlike_har*100:.2f}%")

    # Per-stock breakdown
    stock_ids = test_df["stock_id"].values
    stock_rmspe = {}
    for sid in np.unique(stock_ids):
        mask = stock_ids == sid
        if mask.sum() >= 5:
            stock_rmspe[sid] = rmspe(actual_rv[mask], final_pred_rv[mask])
    sorted_stocks = sorted(stock_rmspe.items(), key=lambda x: x[1])
    print(f"\n  Best 5 stocks by RMSPE:")
    for sid, val in sorted_stocks[:5]:
        print(f"    Stock {sid}: RMSPE={val:.6f}")
    print(f"  Worst 5 stocks by RMSPE:")
    for sid, val in sorted_stocks[-5:]:
        print(f"    Stock {sid}: RMSPE={val:.6f}")

    # ── SHAP values ───────────────────────────────────────────────────
    print(f"  Computing SHAP values...")
    explainer   = shap.TreeExplainer(model_final)
    shap_values = explainer.shap_values(X_test)
    shap_df     = pd.DataFrame(
        shap_values,
        columns=[f"shap_{f}" for f in FINAL_FEATURES]
    )

    # SHAP summary
    shap_summary = pd.DataFrame({
        "feature":       FINAL_FEATURES,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False)
    print(f"\n  Top 10 features by SHAP:")
    print(shap_summary.head(10).to_string(index=False))
    shap_summary.to_parquet(
        STOCK_LGB_DIR / f"shap_summary_fold_{fold_idx}.parquet",
        index=False
    )

    # ── Save predictions ──────────────────────────────────────────────
    pred_df = test_df[["time_id", "stock_id", "cluster_id",
                        "log_rv_fut", "har_pred_log_rv",
                        "har_pred_rv", "har_residual"]].copy()
    pred_df["lgb_residual_pred"] = lgb_residual_pred
    pred_df["final_pred_log_rv"] = final_pred_log_rv
    pred_df["final_pred_rv"]     = final_pred_rv
    pred_df["actual_rv"]         = actual_rv
    pred_df["fold"]              = fold_idx
    pred_df = pd.concat([pred_df.reset_index(drop=True),
                         shap_df.reset_index(drop=True)], axis=1)
    pred_df.to_parquet(pred_path, index=False)
    print(f"  Predictions saved to {pred_path}")

    del train_df, test_df, X_train, y_train, w_train, X_test
    del d_tr, d_es, model_final, shap_values, shap_df
    gc.collect()

    return {
        "predictions": pred_df,
        "metrics": {
            "rmspe_har":    rmspe_har,    "qlike_har":    qlike_har,
            "rmspe_hybrid": rmspe_hybrid, "qlike_hybrid": qlike_hybrid,
        }
    }


# ── Run across all folds ──────────────────────────────────────────────
print("Training stock-level LightGBM across all 5 folds...")
stock_results = []

for fold_idx in range(5):
    print(f"\n══ Fold {fold_idx} ══")
    result = train_stock_lgbm(fold_idx)
    stock_results.append(result)

# ── Aggregate metrics ─────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"Stock-level LightGBM — 5-Fold CV Summary")
print(f"{'═'*60}")

metrics_list = [r["metrics"] for r in stock_results if "metrics" in r]
if metrics_list:
    for metric in ["rmspe_har", "qlike_har", "rmspe_hybrid", "qlike_hybrid"]:
        vals = [m[metric] for m in metrics_list]
        print(f"  {metric:<20s}: {np.mean(vals):.6f} ± {np.std(vals):.6f}")

all_preds = pd.concat(
    [r["predictions"] for r in stock_results],
    ignore_index=True
)
all_preds.to_parquet(
    STOCK_LGB_DIR / "predictions_all_folds.parquet",
    index=False
)
print(f"\n✓ All predictions saved — {all_preds.shape[0]} total rows")
print(f"Cell 10 complete")


Training stock-level LightGBM across all 5 folds...

══ Fold 0 ══
  RAM before fold 0: 2.06 GB
  Loading fold 0 train...
  Loading fold 0 test...
  Running Optuna (10 trials)...
  Best inner RMSPE: 0.399402
  Best params: {
    "learning_rate": 0.04025192252635066,
    "num_leaves": 125,
    "max_depth": 3,
    "min_child_samples": 65,
    "feature_fraction": 0.502314474212375,
    "bagging_fraction": 0.43903095579116774,
    "bagging_freq": 7,
    "reg_alpha": 7.2866537374910445,
    "reg_lambda": 1.7123375973163988
}
  Fitting final model on full train...
[200]	valid_0's rmspe: 0.406269
[400]	valid_0's rmspe: 0.402578
[600]	valid_0's rmspe: 0.403384
  Best iteration: 482

  Fold 0 Results:
    HAR          — RMSPE: 0.506732, QLIKE: -5.567247
    HAR+LightGBM — RMSPE: 0.378027, QLIKE: -5.551450
    RMSPE improvement: 25.40%
    QLIKE improvement: 0.28%

  Best 5 stocks by RMSPE:
    Stock 29: RMSPE=0.162300
    Stock 56: RMSPE=0.171747
    Stock 124: RMSPE=0.172191
    Stock 50: RMSPE